# LLM prompting implementation 

In [1]:

import pandas as pd 
import torch
from sklearn.model_selection import train_test_split
import tqdm

In [2]:

path = '../dataset/training_data_processed.csv'

CLARITY_LABEL = 'Clear Non-Reply'

AMB_CLASSES = ['Implicit', 'Dodging', 'General', 'Deflection', 'Partial/half-answer']
CN_CLASSES = ['Declining to answer', 'Claims Ignorance', 'Clarification']


df = pd.read_csv(path)

print(df['evasion_label'].unique())

['Explicit' 'General' 'Partial/half-answer' 'Dodging' 'Implicit'
 'Deflection' 'Declining to answer' 'Claims ignorance' 'Clarification']


In [5]:

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=16384
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`